In [1]:
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
import threading
import pyaudio
import wave
import librosa
import numpy as np
import torch
from sklearn.preprocessing import OneHotEncoder
import pywt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Define CNN-LSTM model
class CNN_LSTM_Model(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(CNN_LSTM_Model, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm1d(32)  # Add batch normalization layer
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(64)  # Add batch normalization layer
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.lstm = nn.LSTM(64, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension
        x = self.conv1(x)
        x = self.bn1(x)  # Apply batch normalization
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.bn2(x)  # Apply batch normalization
        x = self.relu(x)
        x = self.pool(x)
        x = x.permute(0, 2, 1)  # Reshape for LSTM input (batch_size, seq_len, input_size)
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]  # Take only the output of the last time step
        out = self.fc(lstm_out)
        out = self.softmax(out)
        return out

# Define custom dataset
class SpeechDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def predict_emotion(model, extract_features):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(extract_features, dtype=torch.float32).unsqueeze(0).to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        return predicted.item()



In [2]:
def extract_features(filename):
    y, sr = librosa.load(filename, duration=3, offset=0.5)

    global dwt_level
    # Apply DWT
    #dwt_level=int(input("Enter DWT wavelet level [1/2/3/4/5] "))
    dwt_level=4
    coeffs = pywt.wavedec(y, 'db1', level=dwt_level) # Adjust wavelet and decomposition level as needed
    mfcc_features = []
    
    for coeff in coeffs:
        mfcc_coeff = np.mean(librosa.feature.mfcc(y=coeff.astype(float), sr=sr, n_mfcc=40).T, axis=0)
        mfcc_features.extend(mfcc_coeff)
    return np.array(mfcc_features)

In [3]:
# Define the LSTM model in PyTorch
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.relu(out[:, -1, :])  # Consider only the output of the last time step
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.softmax(out)
        return out


In [ ]:
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
import threading
import pyaudio
import wave
import time  # Import time module to track recording duration

class EmotionPredictionApp:
    def __init__(self, model):
        self.model = model
        self.is_predicting = False  # Flag to track prediction status

        self.root = tk.Tk()
        self.root.title("Emotion Prediction")
        
        # Set window size and center it on the screen
        window_width = 400
        window_height = 500
        screen_width = self.root.winfo_screenwidth()
        screen_height = self.root.winfo_screenheight()
        x_coordinate = ((screen_width - window_width) // 2)
        y_coordinate = ((screen_height - window_height) // 2)
        self.root.geometry(f"{window_width}x{window_height}+{x_coordinate}+{y_coordinate}")

        self.root.configure(bg="white")  # Change background to white
        
        # Load an image to display in the window
        self.logo_image = Image.open("mic button.png")
        self.logo_image = self.logo_image.resize((300, 300))  # Resize the image
        self.logo_photo = ImageTk.PhotoImage(self.logo_image)

        # Create and configure the main frame
        self.main_frame = ttk.Frame(self.root, padding="50")
        self.main_frame.grid(row=0, column=0)

        # Add a label to display the logo
        self.logo_label = ttk.Label(self.main_frame, image=self.logo_photo)
        self.logo_label.grid(row=0, column=0, columnspan=2, pady=10)

        # Add a label to display predicted emotion
        self.emotion_label = ttk.Label(self.main_frame, text="Predicted Emotion: ", font=("Helvetica", 14), foreground="black")  # Set text color to black
        self.emotion_label.grid(row=1, column=0, pady=5, sticky="e")

        # Add a label to display the predicted emotion text
        self.predicted_emotion_label = ttk.Label(self.main_frame, text="", font=("Helvetica", 14), foreground="blue")
        self.predicted_emotion_label.grid(row=1, column=1, pady=5, sticky="w")

        # Add a button to start live prediction
        self.start_button = ttk.Button(self.main_frame, text="Start Prediction", command=self.start_prediction, style="Main.TButton", width=15)
        self.start_button.grid(row=2, column=0, columnspan=2, pady=10, sticky="nsew")  # Center the button and make it span both columns

        # Apply custom styles
        self.style = ttk.Style()
        self.style.configure("Main.TButton", font=("Helvetica", 12), foreground="black", background="#007bff")  # Set text color to black

        # Keep a reference to the image to prevent garbage collection
        self.logo_label.image = self.logo_photo

    def start_prediction(self):
        if not self.is_predicting:  # Check if prediction is not already in progress
            self.start_button.config(state="disabled")  # Disable the start button during prediction
            self.emotion_label.config(text="Listening...")  # Update the status label
            self.is_predicting = True  # Set the prediction flag

            # Run prediction in a separate thread to avoid blocking the GUI
            prediction_thread = threading.Thread(target=self.live_prediction)
            prediction_thread.start()

    def live_prediction(self):
        CHUNK = 1024
        FORMAT = pyaudio.paInt16
        CHANNELS = 1
        RATE = 44100
        RECORD_SECONDS = 10  # Adjust the duration of each recording as needed
        
        p = pyaudio.PyAudio()

        try:
            start_time = time.time()  # Record the start time
            while time.time() - start_time < RECORD_SECONDS:  # Check if the desired duration has elapsed
                stream = p.open(format=FORMAT,
                                channels=CHANNELS,
                                rate=RATE,
                                input=True,
                                frames_per_buffer=CHUNK)

                frames = []

                for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
                    data = stream.read(CHUNK)
                    frames.append(data)

                stream.stop_stream()
                stream.close()

                # Save audio data to a file for processing
                wf = wave.open("temp.wav", 'wb')
                wf.setnchannels(CHANNELS)
                wf.setsampwidth(p.get_sample_size(FORMAT))
                wf.setframerate(RATE)
                wf.writeframes(b''.join(frames))
                wf.close()

                # Extract MFCC features from the recorded audio file
                features = extract_features("temp.wav")
                features = np.expand_dims(features, -1)

                # Predict emotion label
                predicted_label_index = predict_emotion(self.model, features)

                emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Surprised', 'Sad']
                predicted_emotion = emotion_labels[predicted_label_index]

                self.emotion_label.config(text=f"Predicted Emotion: {predicted_emotion}")

            # Re-enable the Start Prediction button after prediction is complete
            self.start_button.config(state="normal")
            self.is_predicting = False  # Reset the prediction flag

        except KeyboardInterrupt:
            p.terminate()

    def run(self):
        self.root.mainloop()

# The rest of your code remains unchanged

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define model parameters
input_size = 1
hidden_size = 256
num_layers = 1
num_classes = 7

# Load trained model
model = LSTMModel(input_size, hidden_size, num_layers, num_classes).to(device)
model.load_state_dict(torch.load(r"C:\Users\Tharunraj_S\Desktop\Major project\codes\Pytorch\DWT and MFCC\LSTMModel\tess\N_MFCC40\level 2 wavelet\500 Epochs\LSTM_model_(best).pth")) 
model.eval()

# Create an instance of the EmotionPredictionApp class and run the GUI
app = EmotionPredictionApp(model)
app.run()


In [ ]:
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
import threading
import pyaudio
import wave
import time  # Import time module to track recording duration

class EmotionPredictionApp:
    def __init__(self, model):
        self.model = model
        self.is_predicting = False  # Flag to track prediction status

        self.root = tk.Tk()
        self.root.title("Emotion Prediction")
        
        # Set window size and center it on the screen
        window_width = 400
        window_height = 500
        screen_width = self.root.winfo_screenwidth()
        screen_height = self.root.winfo_screenheight()
        x_coordinate = ((screen_width - window_width) // 2)
        y_coordinate = ((screen_height - window_height) // 2)
        self.root.geometry(f"{window_width}x{window_height}+{x_coordinate}+{y_coordinate}")

        self.root.configure(bg="white")  # Change background to white
        
        # Load an image to display in the window
        self.logo_image = Image.open("mic button.png")
        self.logo_image = self.logo_image.resize((300, 300))  # Resize the image
        self.logo_photo = ImageTk.PhotoImage(self.logo_image)

        # Create and configure the main frame
        self.main_frame = ttk.Frame(self.root, padding="50")
        self.main_frame.grid(row=0, column=0)

        # Add a label to display the logo
        self.logo_label = ttk.Label(self.main_frame, image=self.logo_photo)
        self.logo_label.grid(row=0, column=0, columnspan=2, pady=10)

        # Add a label to display predicted emotion
        self.emotion_label = ttk.Label(self.main_frame, text="Predicted Emotion: ", font=("Helvetica", 14), foreground="black")  # Set text color to black
        self.emotion_label.grid(row=1, column=0, pady=5, sticky="e")

        # Add a label to display the predicted emotion text
        self.predicted_emotion_label = ttk.Label(self.main_frame, text="", font=("Helvetica", 14), foreground="blue")
        self.predicted_emotion_label.grid(row=1, column=1, pady=5, sticky="w")

        # Add a button to start live prediction
        self.start_button = ttk.Button(self.main_frame, text="Start Prediction", command=self.start_prediction, style="Main.TButton", width=15)
        self.start_button.grid(row=2, column=0, columnspan=2, pady=10, sticky="nsew")  # Center the button and make it span both columns

        # Apply custom styles
        self.style = ttk.Style()
        self.style.configure("Main.TButton", font=("Helvetica", 12), foreground="black", background="#007bff")  # Set text color to black

        # Keep a reference to the image to prevent garbage collection
        self.logo_label.image = self.logo_photo

    def start_prediction(self):
        if not self.is_predicting:  # Check if prediction is not already in progress
            self.start_button.config(state="disabled")  # Disable the start button during prediction
            self.emotion_label.config(text="Listening...")  # Update the status label
            self.is_predicting = True  # Set the prediction flag

            # Run prediction in a separate thread to avoid blocking the GUI
            prediction_thread = threading.Thread(target=self.live_prediction)
            prediction_thread.start()

    def live_prediction(self):
        CHUNK = 1024
        FORMAT = pyaudio.paInt16
        CHANNELS = 1
        RATE = 44100
        RECORD_SECONDS = 10  # Adjust the duration of each recording as needed
        
        p = pyaudio.PyAudio()

        try:
            start_time = time.time()  # Record the start time
            while time.time() - start_time < RECORD_SECONDS:  # Check if the desired duration has elapsed
                stream = p.open(format=FORMAT,
                                channels=CHANNELS,
                                rate=RATE,
                                input=True,
                                frames_per_buffer=CHUNK)

                frames = []

                for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
                    data = stream.read(CHUNK)
                    frames.append(data)

                stream.stop_stream()
                stream.close()

                # Save audio data to a file for processing
                wf = wave.open("temp.wav", 'wb')
                wf.setnchannels(CHANNELS)
                wf.setsampwidth(p.get_sample_size(FORMAT))
                wf.setframerate(RATE)
                wf.writeframes(b''.join(frames))
                wf.close()

                # Extract MFCC features from the recorded audio file
                features = extract_features("temp.wav")

                # Predict emotion label
                predicted_label_index = predict_emotion(self.model, features)

                emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Pleasantly Surprised', 'Sad']
                predicted_emotion = emotion_labels[predicted_label_index]

                self.emotion_label.config(text=f"Predicted Emotion: {predicted_emotion}")

            # Re-enable the Start Prediction button after prediction is complete
            self.start_button.config(state="normal")
            self.is_predicting = False  # Reset the prediction flag

        except KeyboardInterrupt:
            p.terminate()

    def run(self):
        self.root.mainloop()

# The rest of your code remains unchanged

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define model parameters
input_size = 64
hidden_size = 256
num_layers = 1
num_classes = 7

# Load trained model
model = CNN_LSTM_Model(input_size, hidden_size, num_layers, num_classes).to(device)
model.load_state_dict(torch.load(r"C:\Users\Tharunraj_S\Desktop\Major project\codes\Pytorch\DWT and MFCC\CNN_LSTM_Model\tess\N_MFCC40\level 1 wavelet\500 Epochs\LSTM_model_(best).pth")) 
model.eval()

# Create an instance of the EmotionPredictionApp class and run the GUI
app = EmotionPredictionApp(model)
app.run()
